# V_θ Regularisation Sweep — PARF Architecture

## Motivation

The standalone SPLM V_θ regularisation sweep (`vreg_sweep_v_theta_regularisation.ipynb`)
established that breaking the gauge symmetry with a penalty
$\lambda_V \lVert V_\theta\rVert_2^2$ creates genuine bounded energy minima at a
cost of ~60–90 PPL on TinyShakespeare.

PARF adds pair interactions $V_\phi(h_t, h_s)$ that provide a **second channel
of expressivity**.  The hypothesis is that $V_\phi$ can absorb some of the
dynamic range lost by regularising $V_\theta$, resulting in a **smaller PPL
penalty** and a cleaner separation between self-energy ($V_\theta$) and
interaction energy ($\sum V_\phi$).

## Three questions

| # | Question | Measurement |
|---|---------|-------------|
| Q1 | Does PARF's pair potential $V_\phi$ reduce the PPL cost of V_θ regularisation vs standalone SPLM? | Compare val PPL at matching $\lambda_V$ |
| Q2 | Does the bounded $V_\theta$ make the $V_\theta$/$V_\phi$ energy split more interpretable? | Post-training landscape + attractor analysis |
| Q3 | At what $\lambda_V$ does the attractor structure change qualitatively? | K* and GD convergence across the sweep |

## Cell structure

| Cell ID | $\lambda_V$ | Integrator | What it tests |
|---------|-------------|------------|---------------|
| `PR0` | 0 (baseline) | Euler L=8 | Unregularised PARF baseline |
| `PR1` | $10^{-6}$ | Euler L=8 | Weakest regularisation |
| `PR2` | $10^{-4}$ | Euler L=8 | Moderate — match SPLM VR2 |
| `PR3` | $10^{-2}$ | Euler L=8 | Strong regularisation |
| `PR4` | $1$ | Euler L=8 | Very strong |

Run the notebook once per `CELL`; outputs persist on GDrive across sessions.

## 0. Environment setup + cell selector

In [ ]:
CELL = 'PR0'      # one of: 'PR0' | 'PR1' | 'PR2' | 'PR3' | 'PR4'
SEED = 0

REPO_URL        = 'https://github.com/dimitarpg13/semsimula.git'
REPO_BRANCH     = 'main'
COLAB_REPO_PATH = '/content/semsimula'
GDRIVE_OUT_REL  = 'semsimula_vreg'

import os, sys, shutil, subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
print(f'IN_COLAB = {IN_COLAB}')


def _sh(cmd: str) -> None:
    print(f'$ {cmd}')
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0:
        raise RuntimeError(f'command failed (exit {r.returncode}): {cmd}')


if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    GDRIVE_OUT = Path('/content/drive/MyDrive') / GDRIVE_OUT_REL
    GDRIVE_OUT.mkdir(parents=True, exist_ok=True)
    print(f'GDrive output root = {GDRIVE_OUT}')

    REPO_ROOT = Path(COLAB_REPO_PATH)
    if not (REPO_ROOT / '.git').exists():
        if REPO_ROOT.exists():
            shutil.rmtree(REPO_ROOT)
        _sh(
            f'git clone --depth 1 --branch {REPO_BRANCH} '
            f'{REPO_URL} {REPO_ROOT}'
        )
    else:
        try:
            _sh(f'git -C {REPO_ROOT} fetch --depth 1 origin {REPO_BRANCH}')
            _sh(f'git -C {REPO_ROOT} reset --hard origin/{REPO_BRANCH}')
        except RuntimeError as e:
            print(f'WARNING: could not refresh repo ({e}); using existing checkout.')

    DATA_CACHE = GDRIVE_OUT / 'data'
    DATA_CACHE.mkdir(exist_ok=True)
    repo_data_dir = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'data'
    if repo_data_dir.is_symlink():
        repo_data_dir.unlink()
    elif repo_data_dir.exists():
        try:
            repo_data_dir.rmdir()
        except OSError:
            print(f'NOTE: {repo_data_dir} is non-empty; leaving as-is.')
    if not repo_data_dir.exists():
        repo_data_dir.symlink_to(DATA_CACHE, target_is_directory=True)
        print(f'data cache symlink: {repo_data_dir} -> {DATA_CACHE}')

    RESULTS_ROOT = GDRIVE_OUT / 'vreg_sweep_parf'
    RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

    _sh('pip install -q transformers huggingface_hub pyarrow scikit-learn')

else:
    REPO_ROOT = Path.cwd()
    while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'notebooks').exists():
        REPO_ROOT = REPO_ROOT.parent
    if not (REPO_ROOT / 'notebooks').exists():
        raise RuntimeError('Could not locate the semsimula repo root.')
    RESULTS_ROOT = (
        REPO_ROOT / 'notebooks' / 'conservative_arch' / 'parf'
        / 'results' / 'vreg_sweep_parf'
    )
    RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

CONS_ARCH_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch'
SARF_DIR      = CONS_ARCH_DIR / 'sarf_mass_variant'
PARF_DIR      = CONS_ARCH_DIR / 'parf'
ATTRACTOR_DIR = CONS_ARCH_DIR / 'attractor_analysis'
for p in (str(REPO_ROOT), str(CONS_ARCH_DIR), str(SARF_DIR),
          str(PARF_DIR), str(ATTRACTOR_DIR)):
    if p not in sys.path:
        sys.path.insert(0, p)

print(f'\nREPO_ROOT     = {REPO_ROOT}')
print(f'RESULTS_ROOT  = {RESULTS_ROOT}')
print(f'CELL = {CELL!r}  SEED = {SEED}')

## 1. Disable TF32, set seeds, pick device

In [ ]:
import torch
import numpy as np
import torch.nn.functional as F_torch

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision('highest')

torch.manual_seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

if torch.cuda.is_available():
    device = 'cuda'
    torch.cuda.manual_seed_all(SEED)
    cap = torch.cuda.get_device_capability()
    print(f'CUDA: {torch.cuda.get_device_name(0)}  sm_{cap[0]}{cap[1]}  '
          f'mem={torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
elif torch.backends.mps.is_available():
    device = 'mps'
    print('MPS device')
else:
    device = 'cpu'
    print('CPU only — runs will be slow.')
print(f'device = {device}')

## 2. Translate `CELL` switch into config

In [ ]:
VREG_RECIPES = {
    'PR0': dict(lambda_v=0.0,    steps=4000),
    'PR1': dict(lambda_v=1e-6,   steps=4000),
    'PR2': dict(lambda_v=1e-4,   steps=4000),
    'PR3': dict(lambda_v=1e-2,   steps=4000),
    'PR4': dict(lambda_v=1.0,    steps=4000),
}
if CELL not in VREG_RECIPES:
    raise ValueError(f'CELL must be one of {list(VREG_RECIPES)}; got {CELL!r}')

recipe = VREG_RECIPES[CELL]
LAMBDA_V   = recipe['lambda_v']
STEPS      = recipe['steps']

# PARF architecture: TinyShakespeare config (matches train_parf.py shakespeare)
D          = 128
L          = 8
V_HIDDEN   = 128
V_DEPTH    = 3
VOCAB_SIZE = 50257
MAX_LEN    = 256
DT         = 1.0

# V_phi config (structural, matching train_parf.py defaults)
V_PHI_KIND      = 'structural'
V_PHI_D_TYPE    = 16
V_PHI_D_ANGLE   = 8
V_PHI_PHI_HIDDEN  = 32
V_PHI_THETA_HIDDEN = 32
V_PHI_MLP_HIDDEN  = 64

# Training
BATCH      = 16
BLOCK      = 128
LR         = 5e-4
WD         = 0.01
WARMUP     = 200
GRAD_CLIP  = 1.0
EVAL_INTERVAL = 200
EVAL_ITERS    = 40
LOG_INTERVAL  = 50

print(f'Cell {CELL}: lambda_V={LAMBDA_V}  L={L}  dt={DT}  steps={STEPS}')

## 3. Load TinyShakespeare

In [ ]:
from data_module import load_tiny_shakespeare, get_batch

train_ids, val_ids = load_tiny_shakespeare()
print(f'tokens: train={len(train_ids):,}  val={len(val_ids):,}')

## 4. Build PARF model

In [ ]:
from parf.model_parf import PARFLM, PARFConfig
from sarf_mass_variant.model_sarf_mass import causal_cumulative_mean

# Locate or build the logfreq surprisal file
BUNDLED_LOGFREQ = SARF_DIR / 'results' / 'logfreq_surprisal.npy'
DRIVE_LOGFREQ = RESULTS_ROOT / 'logfreq_surprisal_shakespeare.npy'

if BUNDLED_LOGFREQ.exists():
    LOGFREQ_PATH = BUNDLED_LOGFREQ
    print(f'Using bundled logfreq surprisal: {LOGFREQ_PATH}')
elif DRIVE_LOGFREQ.exists():
    LOGFREQ_PATH = DRIVE_LOGFREQ
    print(f'Using Drive-cached logfreq surprisal: {LOGFREQ_PATH}')
else:
    counts = np.bincount(train_ids.astype(np.int64), minlength=VOCAB_SIZE).astype(np.float64)
    p = (counts + 1.0) / (counts.sum() + VOCAB_SIZE)
    surprisal = (-np.log(p)).astype(np.float32)
    LOGFREQ_PATH = DRIVE_LOGFREQ
    LOGFREQ_PATH.parent.mkdir(parents=True, exist_ok=True)
    np.save(LOGFREQ_PATH, surprisal)
    print(f'Built logfreq from train_ids; saved to {LOGFREQ_PATH}')

cfg = PARFConfig(
    vocab_size=VOCAB_SIZE, d=D, max_len=MAX_LEN,
    L=L, v_hidden=V_HIDDEN, v_depth=V_DEPTH, dt=DT,
    v_phi_kind=V_PHI_KIND,
    v_phi_d_type=V_PHI_D_TYPE, v_phi_d_angle=V_PHI_D_ANGLE,
    v_phi_phi_hidden=V_PHI_PHI_HIDDEN,
    v_phi_theta_hidden=V_PHI_THETA_HIDDEN,
    v_phi_mlp_hidden=V_PHI_MLP_HIDDEN,
    mass_mode='logfreq',
    logfreq_path=str(LOGFREQ_PATH),
)

torch.manual_seed(SEED)
model = PARFLM(cfg).to(device)
n_total = sum(p.numel() for p in model.parameters())
n_v_theta = sum(p.numel() for p in model.V_theta.parameters())
n_v_phi = sum(p.numel() for p in model.V_phi.parameters())
print(f'params: total={n_total:,}  V_theta={n_v_theta:,}  V_phi={n_v_phi:,}')
print(f'V_phi kind: {V_PHI_KIND}  L={L}  dt={DT}')

## 5. Training loop with V_θ regularisation

We decompose the forward pass so that `h_L` remains in the computation graph.
PARF's `_stack_forward` already keeps `h_L` live (it runs velocity-Verlet with
`create_graph=True` in training mode), so we call `_embed` → `_stack_forward`
→ logits → loss + V_θ penalty.

In [ ]:
import math, time, json


def lr_at(step):
    if step < WARMUP:
        return LR * (step + 1) / WARMUP
    progress = (step - WARMUP) / max(STEPS - WARMUP, 1)
    return LR * 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))


def forward_with_vreg(model, x, targets, lambda_v):
    """Decomposed forward: embed -> PARF stack -> logits -> NTP + V_θ reg."""
    h0 = model._embed(x)
    h_L, _ = model._stack_forward(h0, x, return_trajectory=False)

    logits = h_L @ model.E.weight.T
    loss_ntp = F_torch.cross_entropy(
        logits.reshape(-1, model.cfg.vocab_size),
        targets.reshape(-1),
    )

    v_reg_value = torch.tensor(0.0, device=x.device)
    if lambda_v > 0:
        xi = causal_cumulative_mean(h_L.detach())
        V_vals = model.V_theta(xi, h_L)
        v_reg_value = (V_vals ** 2).mean()
        loss = loss_ntp + lambda_v * v_reg_value
    else:
        loss = loss_ntp

    return logits, loss, loss_ntp, v_reg_value


@torch.no_grad()
def evaluate():
    model.eval()
    losses = []
    for _ in range(EVAL_ITERS):
        xb, yb = get_batch(val_ids, BATCH, BLOCK, rng)
        x = torch.from_numpy(xb).to(device)
        y = torch.from_numpy(yb).to(device)
        with torch.enable_grad():
            _, loss = model(x, y)
        losses.append(loss.item())
    model.train()
    return float(np.mean(losses))


opt = torch.optim.AdamW(
    model.parameters(), lr=LR, betas=(0.9, 0.95), weight_decay=WD,
)
model.train()

log = []
t0 = time.time()
for step in range(STEPS):
    for g in opt.param_groups:
        g['lr'] = lr_at(step)

    xb, yb = get_batch(train_ids, BATCH, BLOCK, rng)
    x = torch.from_numpy(xb).to(device)
    y = torch.from_numpy(yb).to(device)

    _, loss, loss_ntp, v_reg = forward_with_vreg(model, x, y, LAMBDA_V)

    opt.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    opt.step()

    if (step + 1) % LOG_INTERVAL == 0 or step == 0:
        msg = (f'[{CELL}] step {step + 1:>5}/{STEPS}  '
               f'lr={lr_at(step):.2e}  '
               f'ntp={loss_ntp.item():.4f}  '
               f'v_reg={v_reg.item():.4f}  '
               f'total={loss.item():.4f}  '
               f'wall={time.time() - t0:.0f}s')
        print(msg)

    if (step + 1) % EVAL_INTERVAL == 0 or (step + 1) == STEPS:
        val_loss = evaluate()
        val_ppl = math.exp(val_loss)
        print(f'  >> val_loss={val_loss:.4f}  val_ppl={val_ppl:.2f}')
        log.append({
            'step': step + 1, 'val_loss': val_loss,
            'val_ppl': val_ppl, 'train_loss_ntp': loss_ntp.item(),
            'v_reg': v_reg.item(), 'train_loss_total': loss.item(),
            'lambda_v': LAMBDA_V,
        })

print(f'\n[{CELL}] Training done.  total wall = {time.time() - t0:.0f}s  '
      f'final val_ppl = {log[-1]["val_ppl"]:.2f}')

## 6. Save checkpoint and training log

In [ ]:
import dataclasses

RUN_DIR = RESULTS_ROOT / CELL / f'seed{SEED}'
RUN_DIR.mkdir(parents=True, exist_ok=True)
full_tag = f'vreg_parf_lv{LAMBDA_V:.0e}_L{L}_seed{SEED}'
ckpt_path = RUN_DIR / f'{full_tag}_ckpt_latest.pt'
log_path = RUN_DIR / f'{full_tag}_training_log.jsonl'

torch.save(
    {
        'model_state_dict': model.state_dict(),
        'model_cfg': dataclasses.asdict(cfg),
        'variant': 'parf',
        'cell': CELL,
        'lambda_v': LAMBDA_V,
        'step': STEPS,
        'final_val_ppl': log[-1]['val_ppl'],
        'seed': SEED,
    },
    ckpt_path,
)
with open(log_path, 'w') as f:
    for row in log:
        f.write(json.dumps(row) + '\n')
print(f'wrote ckpt: {ckpt_path}')
print(f'wrote log : {log_path}')

## 7. Plot val PPL trajectory

In [ ]:
import matplotlib.pyplot as plt

steps_log = [r['step'] for r in log]
ppl_log = [r['val_ppl'] for r in log]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(steps_log, ppl_log, marker='o', color='#3a6ea5', label=f'{CELL} (\u03bb={LAMBDA_V})')
ax.set_xlabel('train step')
ax.set_ylabel('val PPL')
ax.set_title(f'{CELL} — PARF val PPL  (final = {ppl_log[-1]:.2f})')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
vreg_log = [r['v_reg'] for r in log]
ax.plot(steps_log, vreg_log, marker='s', color='#b03030', label='mean V_\u03b8\u00b2')
ax.set_xlabel('train step')
ax.set_ylabel('V_\u03b8\u00b2 (regularisation term)')
ax.set_title(f'{CELL} — V_\u03b8 regularisation  (\u03bb_V = {LAMBDA_V})')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RUN_DIR / f'{full_tag}_training.png', dpi=120)
plt.show()

## 8. Post-training attractor extraction

Three attractor-finding protocols, identical to the SPLM VReg sweep:

1. **Pure gradient descent on V_θ** (should converge to genuine minima if λ_V > 0)
2. **Anchored descent** (V_θ + Gaussian prior)
3. **Damped dynamics** (PARF's own integrator at L_train steps)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

try:
    from transformers import GPT2Tokenizer
    tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
except Exception:
    import tiktoken
    enc = tiktoken.get_encoding('gpt2')
    class _Tok:
        def encode(self, s): return enc.encode(s)
        def decode(self, ids): return enc.decode(ids)
    tokenizer = _Tok()

PROMPTS = [
    ('narrative',   'The old king sat on the'),
    ('mathematics', 'The theorem states that for every'),
    ('scientific',  'Photosynthesis converts carbon dioxide and'),
    ('dialogue',    'She whispered: I love'),
    ('code',        'def fibonacci(n): return 1 if n < 2 else'),
]

N_SEEDS = 384
DESCENT_STEPS = 1500
DESCENT_LR = 0.05

model.eval()


def build_seeds_simple(h_mean, h_std, n=384):
    gen = torch.Generator(device='cpu').manual_seed(SEED)
    noise = torch.randn(n, D, generator=gen).to(device)
    return h_mean + noise * h_std


def descend_pure(model, xi, seeds, steps=1500, lr=0.05):
    h = seeds.clone().detach().requires_grad_(True)
    xi_b = xi.unsqueeze(0).expand(h.shape[0], -1)
    opt_d = torch.optim.Adam([h], lr=lr)
    for s in range(steps):
        V = model.V_theta(xi_b, h).sum()
        opt_d.zero_grad()
        V.backward()
        opt_d.step()
        if (s + 1) % max(steps // 5, 1) == 0:
            with torch.no_grad():
                v_mean = V.item() / h.shape[0]
            print(f'  [gd] step {s+1:5d}/{steps}  <V>={v_mean:+.3f}')
    h_f = h.detach().clone().requires_grad_(True)
    V = model.V_theta(xi_b, h_f).sum()
    g, = torch.autograd.grad(V, h_f)
    grad_norm = g.norm(dim=-1).detach().cpu().numpy()
    converged = int((grad_norm < 0.05).sum())
    V_final = model.V_theta(xi_b, h_f.detach()).detach().cpu().numpy().ravel()
    return h.detach().cpu().numpy(), grad_norm, V_final, converged


def descend_anchored(model, xi, seeds, h_center, h_std, lam=0.1,
                     steps=1500, lr=0.05):
    h = seeds.clone().detach().requires_grad_(True)
    xi_b = xi.unsqueeze(0).expand(h.shape[0], -1)
    h_c = h_center.to(device)
    h_s = h_std.to(device)
    opt_d = torch.optim.Adam([h], lr=lr)
    for s in range(steps):
        V = model.V_theta(xi_b, h).sum()
        reg = 0.5 * lam * (((h - h_c) / h_s) ** 2).sum()
        loss_a = V + reg
        opt_d.zero_grad()
        loss_a.backward()
        opt_d.step()
    h_f = h.detach().clone().requires_grad_(True)
    V = model.V_theta(xi_b, h_f).sum()
    reg = 0.5 * lam * (((h_f - h_c) / h_s) ** 2).sum()
    g, = torch.autograd.grad(V + reg, h_f)
    grad_norm = g.norm(dim=-1).detach().cpu().numpy()
    converged = int((grad_norm < 0.05).sum())
    V_final = model.V_theta(xi_b, h_f.detach()).detach().cpu().numpy().ravel()
    return h.detach().cpu().numpy(), grad_norm, V_final, converged


def simulate_damped_simple(model, xi, seeds, n_steps):
    """Semi-implicit Euler from seeds at fixed xi (V_theta only, no V_phi)."""
    h = seeds.clone().detach().to(device)
    v = torch.zeros_like(h)
    xi_b = xi.unsqueeze(0).expand(h.shape[0], -1)
    m_val = float(model.m_global.item())
    gamma_val = float(model.gamma.item())
    dt_val = float(cfg.dt)
    for s in range(n_steps):
        h.requires_grad_(True)
        V = model.V_theta(xi_b, h).sum()
        g, = torch.autograd.grad(V, h)
        h = h.detach()
        f = -g.detach()
        v = (v + dt_val * f / m_val) / (1.0 + dt_val * gamma_val)
        h = h + dt_val * v
    return h.detach().cpu().numpy()


def sweep_clusters(h, K_min=2, K_max=10):
    sils = {}
    best_K, best_sil = K_min, -np.inf
    for K in range(K_min, K_max + 1):
        if len(h) <= K:
            continue
        km = KMeans(n_clusters=K, random_state=SEED, n_init=10).fit(h)
        try:
            sil = silhouette_score(h, km.labels_)
        except Exception:
            sil = -np.inf
        sils[K] = float(sil)
        if sil > best_sil:
            best_sil, best_K = sil, K
    km = KMeans(n_clusters=best_K, random_state=SEED, n_init=10).fit(h)
    return best_K, km.labels_, km.cluster_centers_, sils


def decode_centroids(centroids):
    c_t = torch.from_numpy(centroids).to(device).float()
    logits = c_t @ model.E.weight.T
    probs = torch.softmax(logits, dim=-1)
    results = []
    for i in range(len(centroids)):
        top_vals, top_ids = probs[i].topk(5)
        tokens = [(tokenizer.decode([tid.item()]), float(tv.item()))
                  for tid, tv in zip(top_ids, top_vals)]
        results.append(tokens)
    return results

In [ ]:
h_all = []
for _, prompt in PROMPTS:
    ids = torch.tensor(tokenizer.encode(prompt),
                       device=device, dtype=torch.long).unsqueeze(0)
    with torch.enable_grad():
        out = model(ids, return_trajectory=True)
    h_all.append(out[2][-1][0].to(device))
H_all = torch.cat(h_all, dim=0)
h_mean = H_all.mean(0)
h_std = H_all.std(0).clamp_min(1e-3)

attractor_results = {}

for prompt_name, prompt_text in PROMPTS:
    print(f'\n{"="*60}')
    print(f'Prompt: "{prompt_text}" ({prompt_name})')
    print(f'{"="*60}')

    # Get xi at last token from the model's own forward
    ids = torch.tensor(tokenizer.encode(prompt_text),
                       device=device, dtype=torch.long).unsqueeze(0)
    with torch.enable_grad():
        out = model(ids, return_trajectory=True)
    h_last = out[2][-1][0, -1, :].to(device).detach()
    xi = causal_cumulative_mean(out[2][-1][0:1].to(device))[0, -1, :].detach()

    seeds = build_seeds_simple(h_mean, h_std, n=N_SEEDS)

    print('\n--- Protocol 1: Pure V_\u03b8 descent ---')
    h_gd, gnorm_gd, V_gd, n_conv_gd = descend_pure(
        model, xi, seeds, steps=DESCENT_STEPS, lr=DESCENT_LR)
    print(f'  converged (||\u2207V|| < 0.05): {n_conv_gd}/{N_SEEDS}')
    print(f'  <V> = {V_gd.mean():.1f}  ||h|| = {np.linalg.norm(h_gd, axis=-1).mean():.1f}')
    K_gd, labels_gd, centers_gd, sils_gd = sweep_clusters(h_gd)
    dec_gd = decode_centroids(centers_gd)
    print(f'  K* = {K_gd}  silhouettes = {sils_gd}')
    for i, tokens in enumerate(dec_gd):
        size = (labels_gd == i).sum()
        top3 = ', '.join(f'{repr(t)}: {p:.2f}' for t, p in tokens[:3])
        print(f'    A{i} ({size}): {top3}')

    print('\n--- Protocol 2: Anchored descent (\u03bb_anchor=0.1) ---')
    h_anch, gnorm_anch, V_anch, n_conv_anch = descend_anchored(
        model, xi, seeds, h_mean, h_std, lam=0.1,
        steps=DESCENT_STEPS, lr=DESCENT_LR)
    K_anch, labels_anch, centers_anch, sils_anch = sweep_clusters(h_anch)
    dec_anch = decode_centroids(centers_anch)
    print(f'  K* = {K_anch}  silhouettes = {sils_anch}')
    for i, tokens in enumerate(dec_anch):
        size = (labels_anch == i).sum()
        top3 = ', '.join(f'{repr(t)}: {p:.2f}' for t, p in tokens[:3])
        print(f'    A{i} ({size}): {top3}')

    print(f'\n--- Protocol 3: Damped dynamics ({L} steps) ---')
    h_dyn = simulate_damped_simple(model, xi, seeds, n_steps=L)
    K_dyn, labels_dyn, centers_dyn, sils_dyn = sweep_clusters(h_dyn)
    dec_dyn = decode_centroids(centers_dyn)
    print(f'  K* = {K_dyn}  silhouettes = {sils_dyn}')
    for i, tokens in enumerate(dec_dyn):
        size = (labels_dyn == i).sum()
        top3 = ', '.join(f'{repr(t)}: {p:.2f}' for t, p in tokens[:3])
        print(f'    A{i} ({size}): {top3}')

    attractor_results[prompt_name] = {
        'gd': {'K': K_gd, 'sils': sils_gd, 'decoded': dec_gd,
               'n_converged': n_conv_gd, 'mean_V': float(V_gd.mean()),
               'mean_h_norm': float(np.linalg.norm(h_gd, axis=-1).mean())},
        'anchored': {'K': K_anch, 'sils': sils_anch, 'decoded': dec_anch,
                     'n_converged': n_conv_anch},
        'dyn': {'K': K_dyn, 'sils': sils_dyn, 'decoded': dec_dyn},
    }

attr_path = RUN_DIR / f'{full_tag}_attractor_results.json'
with open(attr_path, 'w') as f:
    json.dump(attractor_results, f, indent=2, default=str)
print(f'\nwrote attractor results: {attr_path}')

## 9. V_θ landscape diagnostics

In [ ]:
v_samples = []
model.eval()
with torch.no_grad():
    for _ in range(10):
        xb, _ = get_batch(val_ids, BATCH, BLOCK, rng)
        x = torch.from_numpy(xb).to(device)
        with torch.enable_grad():
            out = model(x, return_trajectory=True)
        traj = out[2]
        h_L = traj[-1].to(device)
        xi = causal_cumulative_mean(h_L)
        V_vals = model.V_theta(xi, h_L).detach().cpu().numpy().ravel()
        v_samples.append(V_vals)

V_all = np.concatenate(v_samples)
print(f'V_theta on real trajectories (after training with \u03bb_V = {LAMBDA_V}):')
print(f'  mean   = {V_all.mean():.2f}')
print(f'  std    = {V_all.std():.2f}')
print(f'  min    = {V_all.min():.2f}')
print(f'  max    = {V_all.max():.2f}')
print(f'  range  = {V_all.max() - V_all.min():.2f}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(V_all, bins=100, color='#3a6ea5', alpha=0.7, edgecolor='white')
ax.axvline(0, color='red', linestyle='--', alpha=0.5)
ax.set_xlabel('V_\u03b8(\u03be, h)')
ax.set_ylabel('count')
ax.set_title(f'{CELL} \u2014 V_\u03b8 distribution on real trajectories  (\u03bb_V = {LAMBDA_V})')
plt.tight_layout()
plt.savefig(RUN_DIR / f'{full_tag}_v_theta_hist.png', dpi=120)
plt.show()

landscape_stats = {
    'mean': float(V_all.mean()), 'std': float(V_all.std()),
    'min': float(V_all.min()), 'max': float(V_all.max()),
    'range': float(V_all.max() - V_all.min()),
}
with open(RUN_DIR / f'{full_tag}_landscape_stats.json', 'w') as f:
    json.dump(landscape_stats, f, indent=2)

## 10. PCA attractor landscape visualisation

In [ ]:
from sklearn.decomposition import PCA

for prompt_name, prompt_text in PROMPTS:
    ids = torch.tensor(tokenizer.encode(prompt_text),
                       device=device, dtype=torch.long).unsqueeze(0)
    with torch.enable_grad():
        out = model(ids, return_trajectory=True)
    xi = causal_cumulative_mean(out[2][-1][0:1].to(device))[0, -1, :].detach()

    seeds = build_seeds_simple(h_mean, h_std, n=N_SEEDS)

    h_gd, _, V_gd, _ = descend_pure(model, xi, seeds, steps=DESCENT_STEPS, lr=DESCENT_LR)
    h_dyn = simulate_damped_simple(model, xi, seeds, n_steps=L)

    all_h = np.concatenate([h_gd, h_dyn], axis=0)
    pca = PCA(n_components=2).fit(all_h)
    h_gd_2d = pca.transform(h_gd)
    h_dyn_2d = pca.transform(h_dyn)

    K_gd, labels_gd, _, _ = sweep_clusters(h_gd)
    K_dyn, labels_dyn, _, _ = sweep_clusters(h_dyn)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    ax = axes[0]
    for k in range(K_gd):
        mask = labels_gd == k
        ax.scatter(h_gd_2d[mask, 0], h_gd_2d[mask, 1], s=8, alpha=0.5, label=f'A{k}')
    ax.set_title(f'GD basins (K*={K_gd}) \u2014 {prompt_name}')
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
    ax.legend(fontsize=7)

    ax = axes[1]
    for k in range(K_dyn):
        mask = labels_dyn == k
        ax.scatter(h_dyn_2d[mask, 0], h_dyn_2d[mask, 1], s=8, alpha=0.5, label=f'A{k}')
    ax.set_title(f'Damped dynamics basins (K*={K_dyn}) \u2014 {prompt_name}')
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
    ax.legend(fontsize=7)

    plt.suptitle(f'{CELL} (\u03bb_V={LAMBDA_V}) \u2014 PARF attractor landscape: "{prompt_text}"',
                 fontsize=11)
    plt.tight_layout()
    plt.savefig(RUN_DIR / f'{full_tag}_attractors_{prompt_name}.png', dpi=120)
    plt.show()

## 11. Sweep dashboard

Auto-collects results from all PR cells present under `RESULTS_ROOT`.

In [ ]:
results = {}
for cell_name in ('PR0', 'PR1', 'PR2', 'PR3', 'PR4'):
    cell_dir = RESULTS_ROOT / cell_name / f'seed{SEED}'
    if not cell_dir.exists():
        results[cell_name] = None
        continue
    logs = sorted(cell_dir.glob('*_training_log.jsonl'))
    if not logs:
        results[cell_name] = None
        continue
    rows = [json.loads(line) for line in logs[-1].read_text().splitlines()]
    last = rows[-1] if rows else None
    attr_files = sorted(cell_dir.glob('*_attractor_results.json'))
    attr = None
    if attr_files:
        attr = json.loads(attr_files[-1].read_text())
    ls_files = sorted(cell_dir.glob('*_landscape_stats.json'))
    ls = None
    if ls_files:
        ls = json.loads(ls_files[-1].read_text())
    results[cell_name] = {
        'val_ppl': last['val_ppl'] if last else None,
        'lambda_v': VREG_RECIPES[cell_name]['lambda_v'],
        'attractors': attr,
        'landscape': ls,
    }

print(f'{"Cell":<6} {"\u03bb_V":>10} {"val_ppl":>10} '
      f'{"V_\u03b8 range":>10} {"GD conv%":>10} {"K* (GD)":>8} {"K* (anch)":>9} {"K* (dyn)":>8}')
print('-' * 85)
for cell_name, r in results.items():
    if r is None:
        print(f'{cell_name:<6} {"":>10} {"\u2014":>10} '
              f'{"\u2014":>10} {"\u2014":>10} {"\u2014":>8} {"\u2014":>9} {"\u2014":>8}    (not run)')
        continue
    ppl = r['val_ppl']
    lv = r['lambda_v']
    v_range = f"{r['landscape']['range']:.1f}" if r['landscape'] else '\u2014'
    if r['attractors']:
        prompts_a = r['attractors']
        conv_pcts = [prompts_a[p]['gd']['n_converged'] / N_SEEDS * 100
                     for p in prompts_a if 'gd' in prompts_a[p]]
        k_gds = [prompts_a[p]['gd']['K'] for p in prompts_a]
        k_anchs = [prompts_a[p]['anchored']['K'] for p in prompts_a
                   if 'anchored' in prompts_a[p]]
        k_dyns = [prompts_a[p]['dyn']['K'] for p in prompts_a]
        conv_str = f'{np.mean(conv_pcts):.0f}%'
        k_gd_str = f'{np.mean(k_gds):.1f}'
        k_anch_str = f'{np.mean(k_anchs):.1f}' if k_anchs else '\u2014'
        k_dyn_str = f'{np.mean(k_dyns):.1f}'
    else:
        conv_str = k_gd_str = k_anch_str = k_dyn_str = '\u2014'
    ppl_str = f'{ppl:.2f}' if ppl else '\u2014'
    print(f'{cell_name:<6} {lv:>10.0e} {ppl_str:>10} '
          f'{v_range:>10} {conv_str:>10} {k_gd_str:>8} {k_anch_str:>9} {k_dyn_str:>8}')

print(f'\nCompare with standalone SPLM VReg sweep results (vreg_sweep/ on GDrive).')

## 12. Interpretation guide

### Key comparison: PARF vs standalone SPLM

| Outcome | What it means |
|---------|---------------|
| PARF PR0 PPL < SPLM VR0 PPL at same λ_V=0 | Pair interactions help even without regularisation |
| PARF PPL drop at given λ_V < SPLM PPL drop | V_φ absorbs lost V_θ expressivity — confirms the hypothesis |
| K*(GD) higher for PARF | V_φ-mediated interactions create richer attractor topology |
| V_θ range smaller for PARF at λ_V=0 | V_φ is already sharing the energy-representation load |